In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
print(df)

Explore the dataset with head(), shape, missing values, and the target distribution.

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df["target"].value_counts()

Create X and y

In [ ]:
X = df.drop(columns="target")
y = df["target"]

Split the data into 80% training / 20% testing using random_state=42 and stratify=y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Create a pipeline containing:
- StandardScaler
- LogisticRegression(max_iter=5000)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

scaler = StandardScaler()
log_reg = LogisticRegression(max_iter=5000)

pipeline = Pipeline([
    ('scaler', scaler),
    ('log_reg', log_reg)
])



First train the pipeline using Logistic Regression's default C value

In [ ]:
pipeline.fit(X_train, y_train)

Record its test accuracy, precision, recall, and F1 score

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, pipeline.predict(X_test))
precision = precision_score(y_test, pipeline.predict(X_test))
recall = recall_score(y_test, pipeline.predict(X_test))
f1 = f1_score(y_test, pipeline.predict(X_test))

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")      
print(f"F1 Score: {f1:.2f}")

Now use GridSearchCV with 5-fold cross-validation to test

In [ ]:
from sklearn.model_selection import GridSearchCV
C_values = [0.01, 0.1, 1, 10, 100]

param_grid = {"log_reg__C": C_values} # __ conects the pipeline step name with the parameter name

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train, y_train)

Find the best C.

In [ ]:
print("Best C:", grid_search.best_params_["log_reg__C"])
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Evaluate the best model on the untouched test set using the same four metrics

In [ ]:
best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_predictions)
tuned_precision = precision_score(y_test, tuned_predictions)
tuned_recall = recall_score(y_test, tuned_predictions)
tuned_f1 = f1_score(y_test, tuned_predictions)

print("Tuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")

Compare the tuned model against your original model

GridSearchCV selected C = 0.1. The tuned model had slightly lower test accuracy than the default model, showing that the best average cross-validation setting does not always produce a higher score on one test split.

In [ ]:
print("Default Model")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

print(f"\nBest C: {grid_search.best_params_['log_reg__C']}")

print("\nTuned Model")
print(f"Accuracy: {tuned_accuracy:.2f}")
print(f"Precision: {tuned_precision:.2f}")
print(f"Recall: {tuned_recall:.2f}")
print(f"F1 Score: {tuned_f1:.2f}")